# Automatic Parameter Selection

Demonstrates the G2-based automatic parameter selection for the RM1 size threshold.

G2 = normalised(Moran's I) + normalised(intra-variance)

Lower G2 = better segmentation balance (not over- or under-segmented).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from watershed_seg.filters import epsf
from watershed_seg.watershed_input import h_image
from watershed_seg.watershed import vincent_soille
from watershed_seg.merging import rm1
from watershed_seg.evaluation import morans_i, intra_variance, goodness2
from watershed_seg.auto_params import auto_select_rm1_threshold

# Build synthetic image
rng = np.random.default_rng(1)
img = np.zeros((64, 64, 3), dtype=np.float64)
img[:32, :32] = rng.uniform(200, 220, (32, 32, 3))
img[:32, 32:] = rng.uniform(50,  70,  (32, 32, 3))
img[32:, :32] = rng.uniform(100, 120, (32, 32, 3))
img[32:, 32:] = rng.uniform(150, 170, (32, 32, 3))

# Build watershed labels
filtered = epsf(img, w=5)
ws_labels = vincent_soille(h_image(filtered, w=7))
print(f'Watershed: {ws_labels.max()} initial segments')

In [ ]:
# Sweep RM1 threshold and collect Moran's I + variance at each step
search = list(range(1, 41))
all_mi, all_var, seg_counts = [], [], []

for t in search:
    labels = rm1(ws_labels, img, size_threshold=t)
    all_mi.append(morans_i(labels, img))
    all_var.append(intra_variance(labels, img))
    seg_counts.append(int(labels.max()))

mi_arr = np.array(all_mi).T   # C×S
var_arr = np.array(all_var).T # C×S
g2 = goodness2(mi_arr, var_arr)

best_idx = int(np.argmin(g2))
best_thresh = search[best_idx]
print(f'Auto-selected threshold: {best_thresh} (G2={g2[best_idx]:.4f})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(search, g2, 'b-o', markersize=4)
axes[0].axvline(x=best_thresh, color='r', linewidth=2, label=f'Auto-selected={best_thresh}')
axes[0].set_xlabel('RM1 size threshold')
axes[0].set_ylabel('G2 (lower = better)')
axes[0].set_title('G2 curve for RM1 threshold sweep')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(search, seg_counts, 'g-s', markersize=4)
axes[1].axvline(x=best_thresh, color='r', linewidth=2, label=f'Auto-selected={best_thresh}')
axes[1].set_xlabel('RM1 size threshold')
axes[1].set_ylabel('Number of segments')
axes[1].set_title('Segment count vs. threshold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Visual comparison: auto-selected vs. a fixed threshold
manual_thresh = 5
auto_labels   = rm1(ws_labels, img, size_threshold=best_thresh)
manual_labels = rm1(ws_labels, img, size_threshold=manual_thresh)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(ws_labels, cmap='tab20')
axes[0].set_title(f'Watershed ({ws_labels.max()} segs)')
axes[0].axis('off')
axes[1].imshow(manual_labels, cmap='tab20')
axes[1].set_title(f'Manual thresh={manual_thresh} ({manual_labels.max()} segs)')
axes[1].axis('off')
axes[2].imshow(auto_labels, cmap='tab20')
axes[2].set_title(f'Auto thresh={best_thresh} ({auto_labels.max()} segs)')
axes[2].axis('off')
plt.tight_layout()
plt.show()